In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from ipywidgets import interact, FloatSlider, Layout, Button
from IPython.display import display


rng_seed: int = 42
np.random.seed(rng_seed)

last_fig: Optional[plt.Figure] = None

def make_covariance(angle_deg: float, var1: float, var2: float) -> np.ndarray:
    """
    Construct a 2x2 covariance matrix representing an ellipse with principal variances
    var1 and var2, rotated by angle_deg degrees counter-clockwise.

    Parameters
    ----------
    angle_deg : float
        Rotation angle in degrees.
    var1 : float
        Variance along the first principal axis.
    var2 : float
        Variance along the second principal axis.

    Returns
    -------
    np.ndarray
        A 2x2 covariance matrix.
    """
    theta = np.radians(angle_deg)
    R = np.array([[np.cos(theta), -np.sin(theta)],
                  [np.sin(theta),  np.cos(theta)]])
    D = np.diag([var1, var2])
    return R @ D @ R.T

covariances: List[np.ndarray] = [
    make_covariance(45.0, 3.0, 0.5),
    make_covariance(-45.0, 3.0, 0.5),
    make_covariance(45.0, 3.0, 0.5),
]

weights: List[float] = [1.0 / 3.0, 1.0 / 3.0, 1.0 / 3.0]
n_samples: int = 5000

def run_em_and_plot(
    mx0: float,
    my0: float,
    mx1: float,
    my1: float,
    mx2: float,
    my2: float,
) -> None:
    """
    Regenerate mixture samples with a fixed RNG seed given three mean locations,
    fit a 3-component Gaussian Mixture via EM, color points by soft assignments
    ordered along the x-axis (left=red, middle=green, right=blue), and render the plot.

    Parameters
    ----------
    mx0, my0, mx1, my1, mx2, my2 : float
        Coordinates for the three component means.

    Returns
    -------
    None
    """
    global last_fig

    np.random.seed(rng_seed)

    means: List[np.ndarray] = [
        np.array([mx0, my0]),
        np.array([mx1, my1]),
        np.array([mx2, my2]),
    ]

    component_choices = np.random.choice([0, 1, 2], size=n_samples, p=weights)
    samples = np.zeros((n_samples, 2))
    for i in range(3):
        num_i = int(np.sum(component_choices == i))
        if num_i > 0:
            samples[component_choices == i] = np.random.multivariate_normal(
                mean=means[i],
                cov=covariances[i],
                size=num_i,
            )

    gmm = GaussianMixture(
        n_components=3,
        covariance_type="full",
        init_params="kmeans",
        random_state=42,
    )
    gmm.fit(samples)
    probs = gmm.predict_proba(samples)

    sorted_idx = np.argsort(gmm.means_[:, 0])

    color_order = np.zeros((3, 3))
    color_order[sorted_idx[0], 0] = 1.0
    color_order[sorted_idx[1], 1] = 1.0
    color_order[sorted_idx[2], 2] = 1.0

    colors = probs @ color_order

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(samples[:, 0], samples[:, 1], c=colors, s=25)
    ax.scatter(gmm.means_[:, 0], gmm.means_[:, 1], c="black", marker="x", s=120, label="EM Means")
    ax.set_title("EM on Sampled GMM (spatial color ordering)")
    ax.axis("equal")
    ax.grid(True)
    ax.legend()
    plt.show()

    last_fig = fig

def save_screenshot(path: Path, dpi: int = 300) -> None:
    """
    Save the most recently rendered matplotlib Figure to the given path.

    Parameters
    ----------
    path : Path
        Destination file path for the image.
    dpi : int
        Resolution in dots per inch.

    Returns
    -------
    None
    """
    if last_fig is None:
        print("No plot available to save.")
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    last_fig.savefig(str(path), dpi=dpi)
    print(f"Saved plot to {path}")

screenshot_btn = Button(description="Save Screenshot", layout=Layout(width="200px"))
def _on_screenshot_click(_) -> None:
    """
    Button callback to save the current figure to plots/em_algo.png at 300 dpi.

    Returns
    -------
    None
    """
    save_screenshot(Path("plots").joinpath("em_algo.png"), dpi=300)

screenshot_btn.on_click(_on_screenshot_click)
display(screenshot_btn)

sliders = [
    FloatSlider(value=-4.0, min=-10.0, max=10.0, step=0.1, description="μ0 x"),
    FloatSlider(value=-2.0, min=-10.0, max=10.0, step=0.1, description="μ0 y"),
    FloatSlider(value=1.0, min=-10.0, max=10.0, step=0.1, description="μ1 x"),
    FloatSlider(value=-1.0, min=-10.0, max=10.0, step=0.1, description="μ1 y"),
    FloatSlider(value=6.0, min=-10.0, max=10.0, step=0.1, description="μ2 x"),
    FloatSlider(value=-1.0, min=-10.0, max=10.0, step=0.1, description="μ2 y"),
]

interact(
    run_em_and_plot,
    mx0=sliders[0],
    my0=sliders[1],
    mx1=sliders[2],
    my1=sliders[3],
    mx2=sliders[4],
    my2=sliders[5],
);

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from ipywidgets import interact, FloatSlider, Layout, Button
from IPython.display import display


rng_seed: int = 42
np.random.seed(rng_seed)

last_fig: Optional[plt.Figure] = None

def make_covariance(angle_deg: float, var1: float, var2: float) -> np.ndarray:
    """
    Construct a 2x2 covariance matrix representing an ellipse with principal variances
    var1 and var2, rotated by angle_deg degrees counter-clockwise.

    Parameters
    ----------
    angle_deg : float
        Rotation angle in degrees.
    var1 : float
        Variance along the first principal axis.
    var2 : float
        Variance along the second principal axis.

    Returns
    -------
    np.ndarray
        A 2x2 covariance matrix.
    """
    theta = np.radians(angle_deg)
    R = np.array([[np.cos(theta), -np.sin(theta)],
                  [np.sin(theta),  np.cos(theta)]])
    D = np.diag([var1, var2])
    return R @ D @ R.T

covariances: List[np.ndarray] = [
    make_covariance(45.0, 3.0, 0.5),
    make_covariance(-45.0, 3.0, 0.5),
    make_covariance(45.0, 3.0, 0.5),
]

weights: List[float] = [1.0 / 3.0, 1.0 / 3.0, 1.0 / 3.0]
n_samples: int = 5000

def run_em_and_plot(
    mx0: float,
    my0: float,
    mx1: float,
    my1: float,
    mx2: float,
    my2: float,
) -> None:
    """
    Regenerate mixture samples with a fixed RNG seed given three mean locations,
    fit a 3-component Gaussian Mixture via EM, color points by soft assignments
    ordered along the x-axis (left=red, middle=green, right=blue), and render the plot.

    Parameters
    ----------
    mx0, my0, mx1, my1, mx2, my2 : float
        Coordinates for the three component means.

    Returns
    -------
    None
    """
    global last_fig

    np.random.seed(rng_seed)

    means: List[np.ndarray] = [
        np.array([mx0, my0]),
        np.array([mx1, my1]),
        np.array([mx2, my2]),
    ]

    component_choices = np.random.choice([0, 1, 2], size=n_samples, p=weights)
    samples = np.zeros((n_samples, 2))
    for i in range(3):
        num_i = int(np.sum(component_choices == i))
        if num_i > 0:
            samples[component_choices == i] = np.random.multivariate_normal(
                mean=means[i],
                cov=covariances[i],
                size=num_i,
            )

    gmm = GaussianMixture(
        n_components=3,
        covariance_type="full",
        init_params="kmeans",
        random_state=42,
    )
    gmm.fit(samples)
    probs = gmm.predict_proba(samples)

    sorted_idx = np.argsort(gmm.means_[:, 0])

    color_order = np.zeros((3, 3))
    color_order[sorted_idx[0], 0] = 1.0
    color_order[sorted_idx[1], 1] = 1.0
    color_order[sorted_idx[2], 2] = 1.0

    colors = probs @ color_order

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(samples[:, 0], samples[:, 1], c=colors, s=25)
    ax.scatter(gmm.means_[:, 0], gmm.means_[:, 1], c="black", marker="x", s=120, label="EM Means")
    ax.set_title("EM on Sampled GMM (spatial color ordering)")
    ax.axis("equal")
    ax.grid(True)
    ax.legend()
    plt.show()

    last_fig = fig

def save_screenshot(path: Path, dpi: int = 300) -> None:
    """
    Save the most recently rendered matplotlib Figure to the given path.

    Parameters
    ----------
    path : Path
        Destination file path for the image.
    dpi : int
        Resolution in dots per inch.

    Returns
    -------
    None
    """
    if last_fig is None:
        print("No plot available to save.")
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    last_fig.savefig(str(path), dpi=dpi)
    print(f"Saved plot to {path}")

screenshot_btn = Button(description="Save Screenshot", layout=Layout(width="200px"))
def _on_screenshot_click(_) -> None:
    """
    Button callback to save the current figure to plots/em_algo.png at 300 dpi.

    Returns
    -------
    None
    """
    save_screenshot(Path("plots").joinpath("em_algo.png"), dpi=300)

screenshot_btn.on_click(_on_screenshot_click)
display(screenshot_btn)

sliders = [
    FloatSlider(value=-4.0, min=-10.0, max=10.0, step=0.1, description="μ0 x"),
    FloatSlider(value=-2.0, min=-10.0, max=10.0, step=0.1, description="μ0 y"),
    FloatSlider(value=1.0, min=-10.0, max=10.0, step=0.1, description="μ1 x"),
    FloatSlider(value=-1.0, min=-10.0, max=10.0, step=0.1, description="μ1 y"),
    FloatSlider(value=6.0, min=-10.0, max=10.0, step=0.1, description="μ2 x"),
    FloatSlider(value=-1.0, min=-10.0, max=10.0, step=0.1, description="μ2 y"),
]

interact(
    run_em_and_plot,
    mx0=sliders[0],
    my0=sliders[1],
    mx1=sliders[2],
    my1=sliders[3],
    mx2=sliders[4],
    my2=sliders[5],
);